# AMEX Enterprise Credit Risk Platform
## Notebook 26 — Phase 2, Problem 4: Delinquency Escalation / Loss Severity — Business Understanding & LGD Policy
### Problem Statement 4 of 14 (depends on Problem 1's champion PD model)

CRISP-DM stage: **Business Understanding**. First of 4 notebooks for Problem 4 (Notebooks 26-29). Notebook 08 (Phase 1) already computes Expected Credit Loss using a single flat, explicitly-labeled `ASSUMPTION` LGD of 45% for every defaulter, because the Kaggle AMEX dataset carries no real recovery-rate ground truth. Problem 4 replaces that flat number with a real, data-driven **Escalation Severity Score** — computed from each customer's actual D\_\* delinquency-column trend/delta/level trajectory across the real 13-month statement panel (already engineered, unmodified, in Notebook 04's `Feature_Engineering` output) — and validates it against real observed default rates, before assigning each severity tier its own, still-explicitly-labeled `ASSUMPTION` LGD. This tier-differentiated LGD is what Problem 3's ECL engine will consume next.

**Honesty boundary, stated up front:** this dataset has no real loss-severity/recovery-rate ground truth anywhere. What Notebook 27 makes real is the **severity score and its tiering** (100% computed from real features) and the **validation** that tiers rank-order with real observed default rates. What stays an editable `ASSUMPTION` is the **dollar LGD value assigned to each tier** — exactly the same honesty boundary Notebook 08 already drew, just no longer flat.

**Deliverables:** `lgd_policy.json` (read by every later Problem 4 notebook), `p4_stakeholder_analysis.csv`, `LGD_Policy_Charter.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD PROBLEM 1's REAL RESULTS (REQUIRED)
# =============================================================================
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Problem 1's Real Results (Required)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ARTIFACTS = PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction" / "artifacts"
ARTIFACTS_DIR = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem4_Delinquency_Escalation_Loss_Severity" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
NB05_SUMMARY_PATH = P1_ARTIFACTS / "notebook_05_summary.json"
NB08_SUMMARY_PATH = P1_ARTIFACTS / "notebook_08_summary.json"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB05_SUMMARY_PATH, "Problem 4 needs Problem 1's champion PD model -- run Problem 1's Notebook 05 first."),
    (NB08_SUMMARY_PATH, "Problem 4 extends Notebook 08's flat LGD -- run Problem 1's Notebook 08 first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)

RANDOM_SEED = P1_CONFIG["random_seed"]
_resource_limits = P1_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or P1_CONFIG.get("warp_thread_count")
    or P1_CONFIG["hardware"]["logical_cores_detected"]
)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_HOLDOUT_AUC = NB05_SUMMARY["champion_metrics"].get("holdout_auc")
BASELINE_LGD = NB08_SUMMARY["lgd_assumption"]                 # real value written by Notebook 08, not retyped
BASELINE_EAD_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]

print(f"Problem 1 champion model (real)          : {CHAMPION_NAME}")
print(f"Problem 1 champion holdout AUC (real)    : {CHAMPION_HOLDOUT_AUC}")
print(f"Notebook 08 baseline LGD (real, to extend): {BASELINE_LGD:.0%}")
print(f"Notebook 08 baseline EAD/account (real)  : ${BASELINE_EAD_USD:,}")

# --- This problem's own folder skeleton (scaffolded already; self-heal here too) ---
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem4_Delinquency_Escalation_Loss_Severity"
PILLAR_DIRS = {
    "p4_policy": P4_ROOT / "01_LGD_Policy",
    "p4_modeling": P4_ROOT / "02_LGD_Modeling",
    "p4_validation_deployment": P4_ROOT / "03_Validation_Deployment",
    "p4_reporting_packaging": P4_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

# --- This problem's own project_config.json (mirrors Problem 1's real hardware
#     facts -- no re-detection needed, the machine hasn't changed -- plus this
#     problem's own pillar_dirs and data_root). Self-healed/overwritten each run. ---
P4_CONFIG = {
    "project_name": "AMEX Enterprise Credit Risk Platform",
    "config_generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "data_root": P1_CONFIG["data_root"],
    "problem_root": str(P4_ROOT),
    "artifacts_dir": str(ARTIFACTS_DIR),
    "random_seed": RANDOM_SEED,
    "hardware": P1_CONFIG["hardware"],
    "resource_limits": P1_CONFIG.get("resource_limits", {}),
    "warp_thread_count": WARP_THREAD_COUNT,
    "pillar_dirs": {k: str(v) for k, v in PILLAR_DIRS.items()},
}
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(P4_CONFIG, f, indent=2)

print(f"\n\u2705 project_config.json written -- Notebooks 27-29 load this file.")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("(Reporting only -- this notebook defines policy; it does no data-scale work.)")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PROBLEM 4 -- BUSINESS PROBLEM & KPI TREE
# =============================================================================
_section("SECTION 3: Problem 4 -- Business Problem & KPI Tree")

# --- Authored problem framing, anchored to Problem 1/8's real measured values
#     above -- reference/planning documentation, not computed data (same status
#     as Notebook 19's PROBLEM_2_CONTEXT block). ---
PROBLEM_4_CONTEXT = {
    "problem_name": "Delinquency Escalation / Loss Severity",
    "problem_number": 4,
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "depends_on": ["Problem 1: Credit Scoring / PD Prediction (Notebooks 01-18)"],
    "feeds_into": ["Problem 3: Expected Credit Loss (IFRS9/CECL) -- consumes this problem's tier-LGD output"],
    "problem_statement": (
        f"Notebook 08 computes Expected Credit Loss using a single flat LGD of {BASELINE_LGD:.0%} for every "
        "defaulting customer, explicitly flagged as an ASSUMPTION because this dataset has no real "
        "recovery-rate data. Problem 4 replaces the flat number with a real, defensible differentiation: "
        "an Escalation Severity Score computed from each customer's actual D_* trend/delta/level features "
        "(Notebook 04's real per-customer trajectory across the 13-month statement panel), bucketed into "
        "severity tiers, and validated against real observed default rates before each tier is assigned "
        "its own ASSUMPTION LGD value."
    ),
    "kpi_tree": [
        "Severity separation: the real observed default rate of the highest-severity tier must exceed "
        "that of the lowest-severity tier by a wide, statistically significant margin.",
        "Rank-ordering / monotonicity: real observed default rate must increase strictly from lowest to "
        "highest severity tier -- a non-monotonic scheme cannot justify differentiated LGD.",
        "Population balance: no severity tier below the minimum share of the real defaulter population.",
        "Data-driven severity, ASSUMPTION-only dollars: the severity SCORE and its tiering must be 100% "
        "computed from real features -- only the dollar LGD value attached to each tier is an editable "
        "ASSUMPTION, exactly matching Notebook 08's existing honesty boundary.",
    ],
}
print(json.dumps(PROBLEM_4_CONTEXT, indent=2))
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LGD / ESCALATION POLICY -- THE SINGLE SOURCE OF TRUTH FOR NOTEBOOKS 27-29
# =============================================================================
_section("SECTION 4: LGD / Escalation Policy -- The Single Source of Truth for Notebooks 27-29")

# --- Tier count, thresholds, and per-tier LGD are explicit ASSUMPTIONs -- this
#     dataset has no ground truth for real recovery rates. The severity-score
#     construction method (real features only) and the validation KPI targets
#     are the honest, non-fabricated part; Notebook 27 computes and applies both. ---
LGD_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_tiers": 3,
    "tier_order": ["Low Severity", "Moderate Severity", "Severe"],
    "severity_score_method": {
        "description": "ASSUMPTION-free methodology (real data only): for each REAL defaulter (from the "
                        "real labeled default set), an Escalation Severity Score is computed as a weighted "
                        "combination of that customer's real D_* trend_slope, trend_delta, and final-statement "
                        "level features (Notebook 04's real engineered output) -- customers whose D_* signals "
                        "worsened fastest and ended at the worst levels score highest. Notebook 27 computes "
                        "the exact weighting from real feature-target correlation on the training split.",
        "tiering": "ASSUMPTION -- equal-population tertiles (33rd/67th percentile cutpoints) of the real "
                   "severity score distribution, mirroring Notebook 19's quantile tiering approach.",
    },
    "lgd_by_tier": {
        "description": f"ASSUMPTION -- illustrative differentiation around Notebook 08's real flat baseline "
                        f"({BASELINE_LGD:.0%}), not a fitted severity model (no real recovery-rate ground "
                        "truth exists in this dataset to fit one).",
        "values": [
            {"tier": "Low Severity", "tier_order": 1, "lgd": round(BASELINE_LGD * 0.67, 4)},
            {"tier": "Moderate Severity", "tier_order": 2, "lgd": BASELINE_LGD},
            {"tier": "Severe", "tier_order": 3, "lgd": round(min(BASELINE_LGD * 1.44, 0.95), 4)},
        ],
    },
    "ead_per_account_usd": BASELINE_EAD_USD,  # ASSUMPTION, inherited unchanged from Notebook 08
    "kpi_targets": {
        "min_default_rate_ratio_top_to_bottom_tier": 1.5,   # ASSUMPTION
        "min_tier_population_pct": 15.0,                    # ASSUMPTION -- no tier below 15% of defaulters
        "require_strict_monotonicity": True,                 # ASSUMPTION
    },
    "champion_pd_model_used": CHAMPION_NAME,
    "baseline_lgd_source": "Problem 1, Notebook 08 (real value, read programmatically)",
}

lgd_policy_path = PILLAR_DIRS["p4_policy"] / "lgd_policy.json"
with open(lgd_policy_path, "w", encoding="utf-8") as f:
    json.dump(LGD_POLICY, f, indent=2)

print(json.dumps(LGD_POLICY, indent=2))
print(f"\n\u2705 Saved -> {lgd_policy_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: STAKEHOLDER ANALYSIS
# =============================================================================
_section("SECTION 5: Stakeholder Analysis")

STAKEHOLDERS = [
    {"stakeholder": "Finance / Provisioning", "interest": "LGD differentiation directly changes the "
     "quarterly ECL reserve Problem 3 will compute -- needs a defensible, documented methodology."},
    {"stakeholder": "Collections", "interest": "Severity tier is a natural prioritization signal -- "
     "needs tiers that correlate with real observed loss outcomes."},
    {"stakeholder": "Model Risk / Compliance (SR 11-7)", "interest": "Needs the ASSUMPTION boundary "
     "between real severity scoring and illustrative LGD dollars documented before use in any "
     "regulatory-adjacent capital or provisioning calculation."},
    {"stakeholder": "Executive / CFO", "interest": "Needs the dollar impact of moving from a flat to a "
     "tier-differentiated LGD stated in plain financial terms (Notebook 29)."},
]
stakeholder_df = pd.DataFrame(STAKEHOLDERS)
stakeholder_path = PILLAR_DIRS["p4_policy"] / "p4_stakeholder_analysis.csv"
stakeholder_df.to_csv(stakeholder_path, index=False)
print(stakeholder_df.to_string(index=False))
print(f"\u2705 Saved -> {stakeholder_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: WORD REPORT -- LGD_POLICY_CHARTER.DOCX
# =============================================================================
_section("SECTION 6: Word Report -- LGD_Policy_Charter.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 4: Delinquency Escalation / Loss Severity -- Business Understanding & LGD Policy Charter")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Problem Statement", level=1)
doc.add_paragraph(PROBLEM_4_CONTEXT["problem_statement"])

_add_heading(doc, "2. KPI Tree", level=1)
for _k in PROBLEM_4_CONTEXT["kpi_tree"]:
    doc.add_paragraph(_k, style="List Bullet")

_add_heading(doc, "3. Severity Score Methodology (real features only)", level=1)
doc.add_paragraph(LGD_POLICY["severity_score_method"]["description"])
doc.add_paragraph(LGD_POLICY["severity_score_method"]["tiering"])

_add_heading(doc, "4. LGD by Tier (ASSUMPTION)", level=1)
doc.add_paragraph(LGD_POLICY["lgd_by_tier"]["description"])
_add_table_from_df(doc, pd.DataFrame(LGD_POLICY["lgd_by_tier"]["values"]))

_add_heading(doc, "5. KPI Targets (ASSUMPTION)", level=1)
_add_kv_table(doc, LGD_POLICY["kpi_targets"])

_add_heading(doc, "6. Stakeholder Analysis", level=1)
_add_table_from_df(doc, stakeholder_df)

report_path = PILLAR_DIRS["p4_policy"] / "LGD_Policy_Charter.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: VERIFICATION
# =============================================================================
_section("SECTION 7: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Policy defines exactly n_tiers tier names", len(LGD_POLICY["tier_order"]) == LGD_POLICY["n_tiers"])
_check("LGD values strictly increase with severity tier",
       [v["lgd"] for v in LGD_POLICY["lgd_by_tier"]["values"]] ==
       sorted(v["lgd"] for v in LGD_POLICY["lgd_by_tier"]["values"]))
_check("Middle tier LGD matches Notebook 08's real baseline exactly",
       LGD_POLICY["lgd_by_tier"]["values"][1]["lgd"] == BASELINE_LGD)
_check("EAD/account inherited unchanged from Notebook 08",
       LGD_POLICY["ead_per_account_usd"] == BASELINE_EAD_USD)

_expected_files = [lgd_policy_path, stakeholder_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 26 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 26 checks passed.")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: WRITE NOTEBOOK 26 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 8: Write Notebook 26 Summary Artifact")

notebook_26_summary = {
    "notebook": "26_lgd_policy", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 4, "problem_name": "Delinquency Escalation / Loss Severity",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "champion_pd_model": CHAMPION_NAME, "baseline_lgd": BASELINE_LGD, "n_tiers": LGD_POLICY["n_tiers"],
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb26_summary_path = ARTIFACTS_DIR / "notebook_26_summary.json"
with open(nb26_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_26_summary, f, indent=2)
print(f"\u2705 Saved -> {nb26_summary_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 9: Notebook 26 Complete -- Handoff to Notebook 27")

print("NOTEBOOK 26: DELINQUENCY ESCALATION / LOSS SEVERITY -- BUSINESS UNDERSTANDING & LGD POLICY -- COMPLETE")
print(f"  Champion PD model (Problem 1, real) : {CHAMPION_NAME}")
print(f"  Severity tiers defined              : {LGD_POLICY['tier_order']}")
print(f"  LGD by tier (ASSUMPTION)             : {[v['lgd'] for v in LGD_POLICY['lgd_by_tier']['values']]}")
print(f"  Files produced                      : {len(_expected_files) + 1}")
for _p in _expected_files + [nb26_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                       : 27_lgd_modeling.ipynb")
print("\n\u2705 Ready to proceed.")
